---

## Setup

In [1]:
use WWW::HorizonsEphemerisSystem;

In [2]:
%%javascript
require.config({
  paths: {
    d3: "https://d3js.org/d3.v7.min",
    d3_3d: "https://unpkg.com/d3-3d@2.0.2/build/d3-3d"
  }
});

require(["d3", "d3_3d"], function(d3, d3_3d) {
  window.d3 = d3;
  window.d33d = d3_3d || window.d33d || {};
  console.log("d3:", d3.version, "d3-3d exports:", Object.keys(window.d33d));
});

In [3]:
my $background = '#1F1F1F';
my $title-color = 'Ivory';

Ivory

In [4]:
#%js
js-d3-list-line-plot(rand xx 30, :$background)

---

## Artemis II trajectory

[JPL Horizons](https://ssd.jpl.nasa.gov/horizons/) published trajectory data that can be accessed via the Raku package "WWW::HorizonsEphemerisData". A user of that package can ask for the state (position and velocity) of the spacecraft. The data assumes a launch on April 1 which is obviously dependent on finding no reason to delay the launch due to technical issues. JPL Horizons refers to the spacecraft using the numerical identifier "-1024".

Date-time spec for the flight of Artemis II:

In [78]:
my @dates = "2026-04-02 01:58:33", "2026-04-10 23:50:00", "10 m";

[2026-04-02 01:58:33 2026-04-10 23:50:00 10 m]

Same date spec by derive via `DateTime` objects:

In [26]:
my $start = DateTime.new(:2026year, :4month, :2day, :1hour, :59minute, :0timezone);
my $end = DateTime.new(:2026year, :4month, :10day, :23hour, :50minute, :0timezone);
my @dates = [$start, $end].map({ "{.year}-{.month.fmt('%02s')}-{.day.fmt('%02s')} {.hh-mm-ss}" });
@dates .= push('10 m');
say (:@dates);

dates => [2026-04-02 01:59:00 2026-04-10 23:50:00 10 m]


Retrieve the data from Horizons System:

In [46]:
my @artemis-position = horizons-ephemeris-data(
    'state',
    ['@-1024', { center => '399', :@dates }],
    'position',
    :modifier<data>,
);
deduce-type( @artemis-position)

Vector(Struct([Calendar Date (TDB), Date, Vx, Vy, Vz, X, Y, Z], [Str, Str, Num, Num, Num, Num, Num, Num]), 1284)

Summary of the trajectory positions:

In [47]:
sink records-summary(@artemis-position, field-names => <Date X Y Z>)

+---------------------------+-------------------------------+-------------------------------+-------------------------------+
| Date                      | X                             | Y                             | Z                             |
+---------------------------+-------------------------------+-------------------------------+-------------------------------+
| 2461136.054861111 => 1    | Min    => -138580.0159455709  | Min    => -389236.1683782268  | Min    => -43779.62187931302  |
| 2461136.645138889 => 1    | 1st-Qu => -118961.9702674999  | 1st-Qu => -344329.8182021609  | 1st-Qu => -39932.501240256766 |
| 2461137.888194445 => 1    | Mean   => -80703.04832964395  | Mean   => -241649.30999908468 | Mean   => -30015.868619494908 |
| 2461135.388194445 => 1    | Median => -97417.0879929208   | Median => -269429.8772880098  | Median => -34320.931722837595 |
| 2461138.061805556 => 1    | 3rd-Qu => -45419.399240532344 | 3rd-Qu => -146360.01755913495 | 3rd-Qu => -22019.0992181

3D plot of the trajectory:

In [51]:
#%js
js-d3-list-line-plot3d(
    @artemis-position.map(*<X Y Z>), 
    :$background, 
    box-ratios => Whatever, 
    :axes, 
    :!boxed
)

----

## Combined plot

Let us combine the trajectory of Artemis II with that of the Moon and also point flight's starting point.

Moon position (with respect to Earth):

In [45]:
my @moon-position = horizons-ephemeris-data(
    'state',
    ['301', { center => '399', :@dates }],
    'position',
    :modifier<data>,
);
deduce-type( @moon-position)

Vector(Struct([Calendar Date (TDB), Date, Vx, Vy, Vz, X, Y, Z], [Str, Str, Num, Num, Num, Num, Num, Num]), 1284)

Start of Artemis ||:

In [68]:
my @start-point = [[|@artemis-position.head<X Y Z>, 'start', 'point'],]

[[-29158.40503151057 -16459.29846759237 -5124.412474237633 start point]]

Combine data:

In [84]:
my @data = 
    (0, 0, 0, 'Earth', 'point'),
    |@moon-position.map({ [|$_<X Y Z>, 'Moon', 'line'] }), 
    |@artemis-position.map({ [|$_<X Y Z>, 'Artemis', 'line'] }), 
    |@start-point, 
; 
@data = @data.map({ <x y z group type>.Array Z=> $_.Array })».Hash; 
deduce-type(@data)

Vector((Any), 2570)

In [89]:
#%js
js-d3-list-point-plot3d(
        @data,
        background => '#1F1F1F',
        height => 800,
        width => 800,
        margins => %(left => 120, bottom => 40),
        title => 'Artemis II trajectory',
        :30title-font-size,
        box-ratios => Whatever,
        :boxed,
        :!axes,
        :!ticks,
        color-scheme => <SteelBlue Silver Pink Red>
);

----

## Trajectory

[JPL Horizons](https://ssd.jpl.nasa.gov/horizons/) published trajectory data that can be accessed via the Raku package "WWW::HorizonsEphemerisData". A user of that package can ask for the state (position and velocity) of the spacecraft. The data assumes a launch on April 1 which is obviously dependent on finding no reason to delay the launch due to technical issues. JPL Horizons refers to the spacecraft using the numerical identifier "-1024".

In [9]:
my @horizonsEarth = horizons-ephemeris-data(
    'state',
    ['@-1024', {
        center => '399',
        dates => ["2026-04-02 01:58:33", "2026-04-10 23:50:00", "10 m"],
    }],
    'position',
    :modifier<data>,
);

deduce-type(@horizonsEarth)

Vector(Struct([Calendar Date (TDB), Date, Vx, Vy, Vz, X, Y, Z], [Str, Str, Num, Num, Num, Num, Num, Num]), 1284)

In [10]:
sink records-summary(@horizonsEarth)

+---------------------------+-------------------------------+--------------------------------+---------------------------------+-------------------------------+----------------------------------------+-------------------------------+-------------------------------+
| Date                      | Vx                            | Vy                             | Vz                              | X                             | Calendar Date (TDB)                    | Y                             | Z                             |
+---------------------------+-------------------------------+--------------------------------+---------------------------------+-------------------------------+----------------------------------------+-------------------------------+-------------------------------+
| 2461136.637881944 => 1    | Min    => -10.3756380569669   | Min    => -4.699956460903548   | Min    => -0.3540405652243561   | Min    => -138580.3959700764  | A.D. 2026-Apr-10 10:18:33.0000 => 1    | 